# Deep Learning Model for NER
Here we fine-tune a pre-trained Transformer (`distilbert-base-cased`) for Token Classification using Hugging Face's `transformers` library.

In [ ]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import evaluate
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

train_df = pd.read_csv('../nlp_d2_data/train_data_ner.csv')
test_df = pd.read_csv('../nlp_d2_data/test_data_ner.csv')

def df_to_hf_dataset(df):
    grouped = df.groupby('sentence_id').agg({
        'words': list,
        'tags': list
    })
    return Dataset.from_pandas(grouped)

train_ds = df_to_hf_dataset(train_df)
test_ds = df_to_hf_dataset(test_df)

unique_tags = train_df['tags'].unique().tolist()
tag2id = {tag: id for id, tag in enumerate(unique_tags)}
id2tag = {id: tag for id, tag in enumerate(unique_tags)}

print(f"Tags: {unique_tags}")

## Tokenization and Alignment
Transformers use subword tokenization, so we need to align our NER labels with the subword tokens.

In [ ]:
model_checkpoint = "distilbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["words"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples["tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(tag2id[label[word_idx]])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_train = train_ds.map(tokenize_and_align_labels, batched=True)
tokenized_test = test_ds.map(tokenize_and_align_labels, batched=True)

## Training the Model

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint, num_labels=len(unique_tags), id2label=id2tag, label2id=tag2id
)

args = TrainingArguments(
    "ner-distilbert",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# Use seqeval for metrics
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [unique_tags[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [unique_tags[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

## Evaluation & Tiny Test

In [ ]:
# Evaluate on test set
trainer.evaluate()

# Predict on Tiny Test
from transformers import pipeline
ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

with open('../nlp_d2_data/tiny_test.csv', 'r') as f:
    lines = f.readlines()
    for line in lines:
        if line.startswith('#') or len(line.strip()) == 0:
            continue
        text = line.strip()
        preds = ner_pipeline(text)
        
        print("Sentence:", text)
        for p in preds:
            print(f"{p['word']}/{p['entity_group']}", end=" ")
        print("\n")